# Bab 4 — Elbow Method

Notebook pendamping Bab 4 Level 5.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

Setelah mempelajari bab ini, kamu mampu:

- Menjelaskan mengapa jumlah kelompok \( K \) perlu dipilih, dan apa peran Elbow Method.
- Menjalankan alur Elbow Method: menghitung Inertia untuk beberapa K, menggambar kurvanya, lalu membaca "siku".
- Membaca kurva elbow dengan bantuan tabel persentase penurunan.
- Menjelaskan mengapa siku bisa **tidak jelas** atau **tidak tunggal**.
- Memakai heuristik sederhana untuk mencari siku, dan memahami bahwa heuristik itu bisa berubah hasilnya.
- Menjelaskan keterbatasan Elbow Method dan mengapa ia hanya petunjuk, bukan jawaban pasti.
- Memadukan elbow dengan metrik lain untuk memilih K.

## Langkah 0: Pindah ke folder utama proyek

In [ ]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

## Bagian A: Siapkan alat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## Bagian B: Tiga dataset dengan kejelasan berbeda

In [ ]:
# Tiga dataset, semuanya 400 titik
X_clear, _ = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=42)
X_overlap, _ = make_blobs(n_samples=400, centers=4, cluster_std=2.5, random_state=42)
X_uniform = np.random.default_rng(42).uniform(-10, 10, size=(400, 2))

datasets = {
    "4 gerombol jelas": X_clear,
    "4 gerombol tumpang tindih": X_overlap,
    "Tanpa kelompok alami": X_uniform,
}

## Bagian C: Menghitung kurva Inertia

In [ ]:
k_values = list(range(1, 11))


def inertia_curve(X, k_values):
    """Inertia K-Means untuk tiap nilai K."""
    return [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_ for k in k_values]


curves = {name: inertia_curve(X, k_values) for name, X in datasets.items()}

table = pd.DataFrame(curves, index=k_values)
table.index.name = "K"
print(table.round(1).to_string())

## Bagian D: Persentase penurunan tiap langkah

In [ ]:
# Penurunan sebagai persen dari Inertia sebelumnya
drop_pct = table.pct_change().mul(-100)
print(drop_pct.round(1).to_string())

## Bagian E: Menggambar tiga kurva elbow

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, values) in zip(axes, curves.items()):
    ax.plot(k_values, values, marker="o", color=palette[0])
    ax.set_title(name)
    ax.set_xlabel("Jumlah kelompok (K)")
    ax.set_xticks(k_values)
axes[0].set_ylabel("Inertia (WCSS)")
plt.tight_layout()
plt.show()

## Bagian F: Mencari siku dengan jarak ke garis

In [ ]:
def elbow_by_line(k_values, inertias):
    """Heuristik siku: titik yang paling jauh dari garis lurus antara titik pertama dan terakhir.

    Kedua sumbu diskalakan ke rentang 0-1 dulu supaya satuan K dan Inertia setara.
    Ini hanya heuristik pembantu, bukan penentu jumlah kelompok yang benar.
    """
    x = np.array(k_values, dtype=float)
    y = np.array(inertias, dtype=float)
    x = (x - x.min()) / (x.max() - x.min())
    y = (y - y.min()) / (y.max() - y.min())
    start, end = np.array([x[0], y[0]]), np.array([x[-1], y[-1]])
    line = end - start
    line = line / np.linalg.norm(line)
    points = np.column_stack([x, y]) - start
    # jarak tegak lurus titik ke garis = panjang komponen yang tegak lurus garis
    distances = np.abs(points[:, 0] * line[1] - points[:, 1] * line[0])
    return k_values[int(distances.argmax())], distances

## Bagian G: Siku menurut heuristik

In [ ]:
for name, values in curves.items():
    k_elbow, distances = elbow_by_line(k_values, values)
    top_two = np.sort(distances)[-2:][::-1]
    print(f"{name:27s} siku heuristik: K = {k_elbow}   (jarak terbesar {top_two[0]:.3f}, kedua {top_two[1]:.3f})")

## Bagian H: Siku berubah bila rentang K berubah

In [ ]:
# Heuristik yang sama, tetapi rentang K dibuat berbeda
print("Rentang K      Dataset                     Siku heuristik")
for k_max in (6, 10, 15):
    ks = list(range(1, k_max + 1))
    for name, X in datasets.items():
        k_elbow, _ = elbow_by_line(ks, inertia_curve(X, ks))
        print(f"1 sampai {k_max:<4d}  {name:27s} K = {k_elbow}")

## Bagian I: Elbow pada Mall Customers

In [ ]:
df = pd.read_csv("data/mall_customers.csv")
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X_mall = StandardScaler().fit_transform(df[features])

mall_k = list(range(1, 11))
mall_inertia = inertia_curve(X_mall, mall_k)
mall_elbow, _ = elbow_by_line(mall_k, mall_inertia)
print("Siku heuristik Mall Customers: K =", mall_elbow)

# Pembanding: Silhouette (dibahas di Bab 5), tidak bisa dihitung pada K = 1
mall_sil = {k: silhouette_score(X_mall, KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_mall))
            for k in range(2, 11)}
best_sil_k = max(mall_sil, key=mall_sil.get)
print("Silhouette tertinggi pada K =", best_sil_k, f"({mall_sil[best_sil_k]:.3f})")
print({k: round(v, 3) for k, v in mall_sil.items()})

## Bagian J: Menggambar elbow Mall Customers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(mall_k, mall_inertia, marker="o", color=palette[0])
axes[0].plot([mall_k[0], mall_k[-1]], [mall_inertia[0], mall_inertia[-1]],
             linestyle="--", color="gray", label="Garis awal-akhir")
axes[0].scatter([mall_elbow], [mall_inertia[mall_elbow - 1]], s=160, marker="o",
                facecolors="none", edgecolors=palette[4], linewidths=2, label=f"Siku heuristik (K = {mall_elbow})")
axes[0].set_xlabel("Jumlah kelompok (K)")
axes[0].set_ylabel("Inertia (WCSS)")
axes[0].set_title("Elbow pada Mall Customers")
axes[0].set_xticks(mall_k)
axes[0].legend(fontsize=8)

axes[1].plot(list(mall_sil.keys()), list(mall_sil.values()), marker="s", color=palette[1])
axes[1].set_xlabel("Jumlah kelompok (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Pembanding: Silhouette")
axes[1].set_xticks(list(mall_sil.keys()))
plt.tight_layout()
plt.show()

## Latihan

1. Pada `X_clear`, berapa persentase penurunan Inertia dari K = 3 ke K = 4? Apa artinya?
2. Ubah `cluster_std=0.8` menjadi `1.5` pada `X_clear`. Hitung persentase penurunan K = 2 sampai K = 8 dan jalankan `elbow_by_line` untuk K 1 sampai 8.
3. Jalankan `elbow_by_line` pada Mall Customers dengan rentang K 1 sampai 8 dan 1 sampai 15. Apa yang kamu amati?
4. Ganti seed data acak dari 42 menjadi 7 (`default_rng(7)`). Berapa siku heuristik? Apa artinya?
5. Mengapa kita perlu menyamakan skala fitur sebelum menghitung kurva elbow?
6. Mengapa kita tidak boleh mengatakan "Elbow Method membuktikan bahwa ada 5 kelompok"?

Jawaban lengkap ada di halaman bab.

## Ringkasan

- **Elbow Method** memilih K dari titik tekuk pada grafik Inertia terhadap K. Idenya: setelah siku, tambahan kelompok tidak banyak menurunkan Inertia.
- Alurnya: skala data, tentukan rentang K, hitung Inertia, gambar kurva, baca siku (dibantu tabel persentase penurunan), lalu verifikasi.
- Pada empat gerombol jelas, penurunan terbesar (81,2%) ada di K = 4, lalu sekitar 10%. Pada gerombol tumpang tindih dan data acak, siku makin samar.
- **Data acak tanpa kelompok pun menghasilkan kurva yang tampak seperti siku.**
- Heuristik jarak ke garis bisa menjawab K yang berbeda dari jumlah gerombol sebenarnya (3, bukan 4, pada data jelas dengan rentang 1 sampai 10), dan jawabannya berubah bila rentang berubah.
- Pada Mall Customers, siku heuristik dan Silhouette sama-sama menunjuk K = 5 (Silhouette 0,555).
- Elbow Method adalah **petunjuk**, bukan penentu objektif. Padukan dengan metrik lain dan pengetahuan domain.